# 🎵 IPT Recognition — Google Colab

Train your own **Instrumental Playing Techniques** classification model directly in the browser, no local install required.

This notebook mirrors `train.ipynb` but runs everything on Google Colab.

**Before you start**
1. Set the runtime to GPU: `Runtime` ▸ `Change runtime type` ▸ **T4 GPU** (or any GPU).
2. Organize your audio files so that each technique has its own sub-folder:

```
my_data_folder/
├── IPTclass_1/
│   ├── audiofile1.wav
│   └── audiofile2.wav
└── IPTclass_2/
    ├── audiofile1.wav
    └── audiofile2.wav
```

3. Upload `my_data_folder` to your Google Drive (you'll connect it below).

Then run the cells top to bottom. ▶️

## 1. Check GPU

In [ ]:
#@title Verify the GPU is available
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Go to Runtime > Change runtime type > GPU, then re-run.")


## 2. Get the code

In [ ]:
#@title Clone the repository and install requirements
import os

REPO_URL = "https://github.com/nbrochec/ipt_recognition.git"
BRANCH   = "performance"  #@param {type:"string"}

if not os.path.isdir("ipt_recognition"):
    !git clone --branch {BRANCH} {REPO_URL}
%cd ipt_recognition
!pip install -q -r requirements.txt
# Colab pre-installs transformers, which this project does not use. It spams warnings
# because it wants torch>=2.4 while we pin torch==2.2.2, so we remove it.
!pip uninstall -q -y transformers
print("\nDone. Working directory:", os.getcwd())


## 3. Connect your data

Mount your Google Drive, then point to the folder that holds your class sub-folders.
The folder is copied into `data/raw/` where the pipeline expects it.

In [ ]:
#@title Mount Google Drive and import your audio folder
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

#@markdown Path to your data folder inside Google Drive (the folder that contains one sub-folder per IPT class):
drive_data_path = "/content/drive/MyDrive/my_data_folder"  #@param {type:"string"}

#@markdown Name to give this folder inside the project (used as `--train_dir`):
train_soundfiles = "my_data_folder"  #@param {type:"string"}

dest = os.path.join("data", "raw", train_soundfiles)
assert os.path.isdir(drive_data_path), f"Not found: {drive_data_path}"
os.makedirs(os.path.dirname(dest), exist_ok=True)
if os.path.isdir(dest):
    shutil.rmtree(dest)
shutil.copytree(drive_data_path, dest)

classes = sorted(d for d in os.listdir(dest) if os.path.isdir(os.path.join(dest, d)))
print(f"Copied to {dest}")
print(f"Found {len(classes)} class folder(s):", classes)


## 4. Preprocess your dataset

In [ ]:
#@title Preprocessing parameters
run_name = "my_run"  #@param {type:"string"}
#@markdown Sampling rate in Hz (downsampling is recommended):
sr = 24000  #@param {type:"integer"}
#@markdown Classification window length. Use `ms` for milliseconds or `samps` for samples.
#@markdown Larger windows suit longer techniques (e.g. legato).
segment_length = "500 ms"  #@param {type:"string"}


In [ ]:
#@title Run preprocessing
!python preprocess.py --name {run_name} --sampling_rate {sr} --train_dir {train_soundfiles} -seglen "{segment_length}"


## 5. Train your model

In [ ]:
#@title Training parameters
#@markdown `cuda` uses the Colab GPU (recommended). Use `cpu` only if no GPU is available.
device = "cuda"  #@param ["cuda", "cpu"]
#@markdown Model architecture (both work for other instruments too):
model = "flute"  #@param ["flute", "eguitar"]
epochs = 100  #@param {type:"integer"}
#@markdown Stop training after this many epochs without improvement:
early_stopping = 5  #@param {type:"integer"}
#@markdown Online data augmentation (polarity inversion, low/high-pass filters):
online_augment = 0  #@param [0, 1] {type:"raw"}


In [ ]:
#@title Run training
!python train.py --name {run_name} --sampling_rate {sr} -seglen "{segment_length}" --device {device} --model {model} --epochs {epochs} --online_augment {online_augment} --early_stopping {early_stopping}


## 6. Get your trained model

Your run is saved under `runs/<run_name>_<date_time>/` and contains the checkpoint
(`.pth`), the TorchScript model (`.ts`) for use in Max/MSP with `ipt~`, and the config.

The cell below finds the latest run and copies it to your Google Drive so it survives
after the Colab session ends.

In [ ]:
#@title Save the latest run to Google Drive
import glob, os, shutil

runs = sorted(glob.glob("runs/*"), key=os.path.getmtime)
assert runs, "No runs found. Did training finish?"
latest = runs[-1]
print("Latest run:", latest)
for f in sorted(os.listdir(latest)):
    print("  -", f)

#@markdown Destination folder in your Google Drive:
drive_output_dir = "/content/drive/MyDrive/ipt_runs"  #@param {type:"string"}
os.makedirs(drive_output_dir, exist_ok=True)
dest = os.path.join(drive_output_dir, os.path.basename(latest))
if os.path.isdir(dest):
    shutil.rmtree(dest)
shutil.copytree(latest, dest)
print("\nSaved to:", dest)


## 7. (Optional) Fine-tune an existing model

Instead of training from scratch, you can **fine-tune** a pretrained model on your own
classes. This adapts the model to your data while reusing what it already learned, which
usually needs far less data and fewer epochs.

By default this uses the bundled `pretrained/flute_pretrained.pth` (`base` architecture),
but you can point to any `.pth` checkpoint — including one you saved to Google Drive in a
previous run.

**Important — audio parameters must match the pretrained model.** Preprocess your
fine-tuning data with the *same* `--sampling_rate` and segment length that the pretrained
model was trained with. `finetune.py` verifies this and will stop if they don't match.

The steps below mirror sections 3–5, but for the fine-tuning workflow.

In [ ]:
#@title Import your fine-tuning audio folder
#@markdown Organize it exactly like in section 3: one sub-folder per IPT class.
from google.colab import drive
import os, shutil

# Mount Drive if it isn't already (no-op if you ran section 3).
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')

#@markdown Path to your fine-tuning data folder inside Google Drive:
ft_drive_data_path = "/content/drive/MyDrive/my_finetune_folder"  #@param {type:"string"}

#@markdown Name to give this folder inside the project (used as `--train_dir`):
ft_train_soundfiles = "my_finetune_folder"  #@param {type:"string"}

ft_dest = os.path.join("data", "raw", ft_train_soundfiles)
assert os.path.isdir(ft_drive_data_path), f"Not found: {ft_drive_data_path}"
os.makedirs(os.path.dirname(ft_dest), exist_ok=True)
if os.path.isdir(ft_dest):
    shutil.rmtree(ft_dest)
shutil.copytree(ft_drive_data_path, ft_dest)

ft_classes = sorted(d for d in os.listdir(ft_dest) if os.path.isdir(os.path.join(ft_dest, d)))
print(f"Copied to {ft_dest}")
print(f"Found {len(ft_classes)} class folder(s):", ft_classes)


In [ ]:
#@title Preprocess the fine-tuning dataset
#@markdown Use the **same** sampling rate and segment length as the pretrained model.
#@markdown The bundled `flute_pretrained.pth` was trained at 44100 Hz with a 14700 samps window.
ft_run_name = "my_finetune_run"  #@param {type:"string"}
ft_sr = 44100  #@param {type:"integer"}
ft_segment_length = "14700 samps"  #@param {type:"string"}

!python preprocess.py --name {ft_run_name} --sampling_rate {ft_sr} --train_dir {ft_train_soundfiles} -seglen "{ft_segment_length}"


In [ ]:
#@title Fine-tuning parameters
#@markdown `cuda` uses the Colab GPU (recommended). Use `cpu` only if no GPU is available.
ft_device = "cuda"  #@param ["cuda", "cpu"]

#@markdown Path to the pretrained checkpoint (`.pth`). Keep the default to use the bundled
#@markdown flute model, or point to one you saved to Google Drive in a previous run.
ft_pretrained = "pretrained/flute_pretrained.pth"  #@param {type:"string"}

#@markdown Architecture of the pretrained model — must match the checkpoint above.
#@markdown The bundled `flute_pretrained.pth` uses `base`.
ft_model = "base"  #@param ["base", "flute", "eguitar"]

#@markdown Freeze the convolutional layers and train only the classifier (recommended for
#@markdown small datasets). Set to 0 to fine-tune the whole network.
freeze_conv = 1  #@param [0, 1] {type:"raw"}

#@markdown Learning rate (use a low value for fine-tuning):
ft_learning_rate = 0.0003  #@param {type:"number"}
ft_epochs = 50  #@param {type:"integer"}
#@markdown Stop after this many epochs without improvement:
ft_early_stopping = 5  #@param {type:"integer"}
#@markdown Online data augmentation (polarity inversion, low/high-pass filters):
ft_online_augment = 0  #@param [0, 1] {type:"raw"}


In [ ]:
#@title Run fine-tuning
!python finetune.py --name {ft_run_name} --pretrained "{ft_pretrained}" --model {ft_model} \
  --sampling_rate {ft_sr} -seglen "{ft_segment_length}" --device {ft_device} \
  --freeze_conv {freeze_conv} --learning_rate {ft_learning_rate} --epochs {ft_epochs} \
  --early_stopping {ft_early_stopping} --online_augment {ft_online_augment}


In [ ]:
#@title Save the fine-tuned run to Google Drive
import glob, os, shutil

ft_runs = sorted(glob.glob("runs/*_finetune_*"), key=os.path.getmtime)
assert ft_runs, "No fine-tuning runs found. Did fine-tuning finish?"
ft_latest = ft_runs[-1]
print("Latest fine-tuning run:", ft_latest)
for f in sorted(os.listdir(ft_latest)):
    print("  -", f)

#@markdown Destination folder in your Google Drive:
ft_drive_output_dir = "/content/drive/MyDrive/ipt_runs"  #@param {type:"string"}
os.makedirs(ft_drive_output_dir, exist_ok=True)
ft_out = os.path.join(ft_drive_output_dir, os.path.basename(ft_latest))
if os.path.isdir(ft_out):
    shutil.rmtree(ft_out)
shutil.copytree(ft_latest, ft_out)
print("\nSaved to:", ft_out)
